# MPQA reproduction: run everything from here

**Before you start (Kaggle):** Settings -> Accelerator **GPU T4 x2**, **Internet: on** (the pretrained ViT weights are
downloaded), and add the MRIQP dataset as an input.

Run the cells top to bottom. Only **step 1** needs editing. Steps marked *(optional)* can be skipped.

| Step | What | Time (T4 x2, rough) |
|---|---|---|
| 1 | setup | 1 min |
| 2 | sanity checks: leakage + metadata-only baseline *(optional)* | seconds, CPU |
| 3 | head-variant smoke test *(optional)* | about 25 min |
| 4 | **train** + test | about 6 min per epoch |
| 5 | evaluate / cleaned test set | 2 min |
| 6 | diagnose per-dimension scores *(optional)* | 2-5 min |
| 7 | training curves | seconds |
| 8 | baselines from the paper's Table 2 *(optional)* | minutes to an hour |

## 1. Setup (edit the first block)

In [ ]:
import os, subprocess, sys

# ---- edit these -------------------------------------------------------------------------------
REPO_URL = "https://github.com/<your-user>/mpqa-repro.git"   # where you pushed the repo
DATA_DIR = "/kaggle/input/datasets/hakimdev/mriqp4/LabelMRI4"  # folder with the CSVs, noisedImages/, originalImages/
OUT_DIR  = "/kaggle/working/run"                               # checkpoints + history.csv are written here
EPOCHS   = 40          # TOTAL epochs (paper: 100). ~6 min/epoch on one T4 -> keep a commit under ~10 h
EXTRA    = ""          # any extra overrides for every command, e.g. "--set batch_size=16 --set lam=2"
# -----------------------------------------------------------------------------------------------

if not os.path.exists("main.py"):                              # not inside the repo yet -> clone it
    subprocess.run(["git", "clone", "-q", REPO_URL, "mpqa-repro"], check=True)
    os.chdir("mpqa-repro")
print("working directory:", os.getcwd())

import shlex

def run(command, *args):
    """Run `python main.py <command> ...` with THIS notebook's Python, stream the output, stop if it fails."""
    p = subprocess.Popen([sys.executable, "-u", "main.py", command, *args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"'{command}' failed with exit code {p.returncode} (see the output above)")

DATA = ["--set", f"data_dir={DATA_DIR}"]
OUT = ["--set", f"out_dir={OUT_DIR}"]
EXTRA_ARGS = shlex.split(EXTRA)

# torch is preinstalled on Kaggle; these are the small extras
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "pyyaml", "scikit-learn", "scikit-image"], check=True)

import torch
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("data folder:", sorted(os.listdir(DATA_DIR))[:8])

## 2. Sanity checks *(optional, CPU, seconds)*
Leakage between train / validation / test, and a baseline that sees **only the degradation tags in the filenames**
(no pixels). All leakage numbers should be about zero; the metadata-only score tells you how much of the benchmark is
explained by *which* degradations were applied.

In [ ]:
run("sanity", *DATA, *EXTRA_ARGS)

## 3. Head-variant smoke test *(optional, about 25 min on 2 GPUs)*
Four short trainings (activation relu/gelu x head dropout 0.5/0) compared on **validation** accuracy per dimension.
The first one (`relu_d0.5`) is the authors' setup and should collapse; the verdict at the end says which variant to use.
With 2 GPUs two variants run at the same time.

In [ ]:
run("smoke", *DATA, *EXTRA_ARGS)

## 4. Train, then test the best checkpoint
Validation runs every epoch (accuracy per dimension is printed). `best.pth` = best validation Overall, `last.pth` = full
state written after every epoch. **If the run stops, just run this cell again: it resumes from `last.pth`.** A failing step stops the notebook with an error.
`EPOCHS` is the total, so raising it later continues the same run.

In [ ]:
run("train", *DATA, *OUT, "--set", f"epochs={EPOCHS}", *EXTRA_ARGS)

## 5. Evaluate a checkpoint
First on the test set as shipped, then with the subject that appears in both CSVs removed from the test set.

In [ ]:
run("evaluate", "--ckpt", f"{OUT_DIR}/best.pth", *DATA, *OUT, *EXTRA_ARGS)
print("\n------ same, with subjects shared with the train CSV removed from the test set ------")
run("evaluate", "--ckpt", f"{OUT_DIR}/best.pth", *DATA, *OUT, "--set", "drop_shared_subjects=true", *EXTRA_ARGS)

## 6. Diagnose the per-dimension scores *(optional)*
Do the SNR / Contrast / Resolution / Artifacts scores really differ between the two images of a pair? A healthy model
has `pct_diff_exactly_0` near 0 and `acc_non_tied` well above 50 in every row.

In [ ]:
run("diagnose", "--ckpt", f"{OUT_DIR}/best.pth", *DATA, *OUT, "--n", "500", *EXTRA_ARGS)

## 7. Training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

h = pd.read_csv(f"{OUT_DIR}/history.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="validation")
ax[0].set_xlabel("epoch"); ax[0].set_title("loss"); ax[0].legend()
for col in ["val_SNR", "val_Contrast", "val_Resolution", "val_Artifacts", "val_Overall"]:
    ax[1].plot(h.epoch, h[col], label=col.replace("val_", ""), lw=3 if col == "val_Overall" else 1.2)
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("accuracy on non-tied pairs (%)"); ax[1].set_title("validation accuracy"); ax[1].legend()
plt.tight_layout(); plt.show()
h.tail(3)

## 8. Baselines from the paper's Table 2 *(optional)*
Compares PSNR, SSIM, LPIPS, DISTS, ILNIQE, LIQE, NIQE, BRISQUE (via `pyiqa`) plus simple SNR / CNR with the paper's
numbers on the **same test pairs**. Try a quick run first. Put your MPQA test Overall in `--mpqa-score` to add its row.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyiqa"], check=True)

In [ ]:
run("baselines", *DATA, "--n-pairs", "100", *EXTRA_ARGS)

In [ ]:
run("baselines", *DATA, "--mpqa-score", "86.83", *EXTRA_ARGS)